# Chapter 03 실습 기록 — 데이터의 첫인상 읽기

## 0. 제출 정보

| 항목 | 내용 |
|---|---|
| 이름 | 김종은 |
| GitHub ID | risa0122 |
| 작성일 | 2026-09-23 |
| 환경 | macOS, Python 3.12.14, pandas 3.0.5, VS Code/Jupyter |
| 최종 제출 URL | https://github.com/risa0122/llm-data-analysis-study/blob/main/chapter03/chapter03.ipynb |

Chapter 02에서 생성한 가상 쇼핑몰 CSV 4개를 사용했다. 데이터 구조와 품질을 확인하고, 관찰한 사실과 해석을 구분해 정리했다. 아래 미리보기의 이름은 Faker로 생성한 가상 값이다.

자료: [공식 답안 양식](https://github.com/GilbertMoon/llm-data-analysis-course/blob/main/practice/chapter03/templates/chapter03_assignment.md), [실습 가이드](https://github.com/GilbertMoon/llm-data-analysis-course/blob/main/practice/chapter03/chapter03.md).

## 1. 데이터 로딩과 구조 확인

### 실행/결과

CSV 파일 존재 여부, 행·열 수, 실제 컬럼명과 타입을 확인했다.

In [1]:
from pathlib import Path
import hashlib
import sys
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 100)

def find_project_root(start):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data/raw").is_dir() and (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("공식 강의 저장소의 chapter03 폴더에서 Notebook을 실행해 주세요.")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data/raw"
names = ["customers", "products", "orders", "order_items"]
file_check = pd.DataFrame({"file": [n + ".csv" for n in names],
                           "exists": [(DATA_DIR / (n + ".csv")).is_file() for n in names]})
assert file_check["exists"].all(), "CSV 파일 존재 여부를 먼저 확인해 주세요."
datasets = {n: pd.read_csv(DATA_DIR / (n + ".csv")) for n in names}
customers, products, orders, order_items = (datasets[n] for n in names)
print("Python:", sys.version.split()[0])
print("실행 파일:", sys.executable)
print("pandas:", pd.__version__)
print("작업 폴더:", Path.cwd())
print("데이터 폴더 존재:", DATA_DIR.exists())
display(file_check)

Python: 3.12.14
실행 파일: /Users/jongeun/llm-data-analysis-course/.venv/bin/python
pandas: 3.0.5
작업 폴더: /Users/jongeun/llm-data-analysis-course/chapter03
데이터 폴더 존재: True


,file,exists
0,customers.csv,True
1,products.csv,True
2,orders.csv,True
3,order_items.csv,True


In [2]:
# 입력 데이터 식별 기록. 이후 재생성한 데이터와 구분한다.
data_fingerprints = {n + ".csv": hashlib.sha256((DATA_DIR / (n + ".csv")).read_bytes()).hexdigest()
                     for n in names}
display(pd.DataFrame(data_fingerprints.items(), columns=["file", "sha256"]))

,file,sha256
0,customers.csv,a663be98960246a436ce7e1ad81284760000ac5ba0d52a7adb3465e6b0457550
1,products.csv,46ff06753ade3e71011ee08dd366148dcd9c36521f111840cead0a9d1047e405
2,orders.csv,0df461755b933af9f5413427a1c160ca2101b24a5fcd2c4f68d5249a90b9a5e3
3,order_items.csv,a7a5c3942b0886c66cb38cc6a10fdfa68dbcde90e8642585dee49c4bd104063a


In [3]:
# CAPTURE 1: 네 파일의 규모와 실제 컬럼
structure = pd.DataFrame([
    {"파일": n, "행": len(df), "열": len(df.columns), "컬럼": ", ".join(df.columns)}
    for n, df in datasets.items()
])
display(structure)
print("날짜 변환 전:", "signup_date =", customers["signup_date"].dtype,
      ", order_date =", orders["order_date"].dtype)

,파일,행,열,컬럼
0,customers,150,6,"customer_id, name, gender, age, city, signup_date"
1,products,100,4,"product_id, product_name, category, price"
2,orders,300,5,"order_id, customer_id, order_date, payment_method, order_status"
3,order_items,764,5,"order_item_id, order_id, product_id, quantity, unit_price"


날짜 변환 전: signup_date = str , order_date = str


In [4]:
for name, df in datasets.items():
    print(name, "head")
    display(df.head())
print("customers tail")
display(customers.tail())

customers head


,customer_id,name,gender,age,city,signup_date
0,1,김수민,F,19,광주,2024-05-17
1,2,장춘자,F,32,대구,2023-12-20
2,3,김상현,F,61,성남,2025-03-22
3,4,김재호,F,55,울산,2025-05-05
4,5,최준서,F,19,부산,2023-09-23


products head


,product_id,product_name,category,price
0,1,전자기기 상품 001,전자기기,160000
1,2,도서 상품 002,도서,34000
2,3,전자기기 상품 003,전자기기,152000
3,4,생활용품 상품 004,생활용품,70000
4,5,식품 상품 005,식품,186000


orders head


,order_id,customer_id,order_date,payment_method,order_status
0,1,123,2025-10-16,card,completed
1,2,77,2026-06-05,naver_pay,cancelled
2,3,138,2026-03-12,bank_transfer,cancelled
3,4,57,2026-06-19,kakao_pay,cancelled
4,5,125,2026-05-25,card,cancelled


order_items head


,order_item_id,order_id,product_id,quantity,unit_price
0,1,1,100,3,102000
1,2,1,87,5,25000
2,3,1,7,3,142000
3,4,1,9,3,193000
4,5,2,72,4,189000


customers tail


,customer_id,name,gender,age,city,signup_date
145,146,고준영,M,61,성남,2023-12-19
146,147,김예은,M,19,부산,2025-07-15
147,148,김준혁,M,29,고양,2024-04-02
148,149,안성현,M,20,부산,2026-05-07
149,150,이경숙,M,40,대전,2026-07-21


In [5]:
for name, df in datasets.items():
    print(name, "컬럼별 타입")
    print(df.dtypes)
customers.info()

customers 컬럼별 타입
customer_id    int64
name             str
gender           str
age            int64
city             str
signup_date      str
dtype: object
products 컬럼별 타입
product_id      int64
product_name      str
category          str
price           int64
dtype: object
orders 컬럼별 타입
order_id          int64
customer_id       int64
order_date          str
payment_method      str
order_status        str
dtype: object
order_items 컬럼별 타입
order_item_id    int64
order_id         int64
product_id       int64
quantity         int64
unit_price       int64
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  150 non-null    int64
 1   name         150 non-null    str  
 2   gender       150 non-null    str  
 3   age          150 non-null    int64
 4   city         150 non-null    str  
 5   signup_date  150 non-null    str  
dtypes: int64(2)

### 결과 관찰

4개 CSV가 모두 존재했다. customers는 150행 6열, products는 100행 4열, orders는 300행 5열, order_items는 764행 5열이었다. signup_date와 order_date는 처음 읽을 때 `str` 타입이었다. age·price·quantity·unit_price와 ID는 정수형으로 읽혔다.

### 나의 해석과 판단

주문 상세는 한 주문에 여러 상품이 들어갈 수 있어 주문보다 행이 많다. ID는 숫자로 저장돼 있어도 식별자이므로 평균을 분석하지 않았다. 날짜는 문자열 상태에서 계산하지 않고 날짜 타입으로 변환해 확인해야 한다고 판단했다.

### 업무·분석적 의미

고객·주문·상품·주문 상세의 행 단위를 구분해야 한다. 주문 상세 행 수를 주문 수로 계산하거나 잘못 병합하면 주문 수와 금액이 부풀려질 수 있다.

### 한계와 추가 확인 사항

head와 tail은 일부 행만 보여 준다. 미리보기와 타입만으로 전체 데이터의 결측, 중복, 날짜 순서가 정상이라고 판단할 수 없다.

### Evidence

캡처 첨부 대기: `step01_structure.png`

<!-- ![데이터 구조 확인](images/step01_structure.png) -->

## 2. 결측·중복·키 품질

### 실행/결과

In [6]:
key_columns = {"customers": "customer_id", "products": "product_id",
               "orders": "order_id", "order_items": "order_item_id"}

def check_data_overview(name, df, key_column):
    return {"파일": name, "결측값": int(df.isna().sum().sum()),
            "전체행 중복": int(df.duplicated().sum()),
            "기본키": key_column, "키 결측": int(df[key_column].isna().sum()),
            "키 중복": int(df[key_column].duplicated().sum())}

# CAPTURE 2: 결측·전체행 중복·기본키 결측 및 중복
quality = pd.DataFrame([check_data_overview(n, df, key_columns[n])
                        for n, df in datasets.items()])
display(quality)
print("order_items.order_id 반복 행 수:", int(order_items["order_id"].duplicated().sum()))
print("order_items.order_id 고유 개수:", order_items["order_id"].nunique())

,파일,결측값,전체행 중복,기본키,키 결측,키 중복
0,customers,0,0,customer_id,0,0
1,products,0,0,product_id,0,0
2,orders,0,0,order_id,0,0
3,order_items,0,0,order_item_id,0,0


order_items.order_id 반복 행 수: 464
order_items.order_id 고유 개수: 300


In [7]:
missing_detail = pd.concat([
    pd.DataFrame({"파일": n, "컬럼": df.columns,
                  "결측 수": df.isna().sum().to_numpy(),
                  "결측 비율(%)": (df.isna().mean() * 100).round(2).to_numpy()})
    for n, df in datasets.items()
], ignore_index=True)
display(missing_detail)

,파일,컬럼,결측 수,결측 비율(%)
0,customers,customer_id,0,0.0
1,customers,name,0,0.0
2,customers,gender,0,0.0
3,customers,age,0,0.0
4,customers,city,0,0.0
5,customers,signup_date,0,0.0
6,products,product_id,0,0.0
7,products,product_name,0,0.0
8,products,category,0,0.0
9,products,price,0,0.0


### 결과 관찰

모든 컬럼의 결측 수와 결측 비율은 0이었다. 네 파일의 전체 행 중복과 각 기본키의 결측·중복도 모두 0이었다. 반면 order_items.order_id의 반복 행 수는 464건이고, 고유한 order_id는 300개였다.

### 나의 해석과 판단

현재 검사 결과에서는 결측 대체나 중복 삭제가 필요하지 않았다. order_items의 기본키는 order_item_id이고 order_id는 연결용 키다. order_id가 반복된다는 이유로 삭제하면 같은 주문의 다른 상품 내역을 잃게 된다.

### 업무·분석적 의미

결측, 전체 행 중복, 기본키 중복을 따로 확인하면 데이터 단위를 보존하면서 품질 문제를 구분할 수 있다. 우선 기본키 결측·중복을 확인하고, 정상 반복인 연결용 키는 유지해야 한다.

### 한계와 추가 확인 사항

값이 존재하고 중복이 없어도 내용이 정확하다는 뜻은 아니다. 비정상적인 범위나 가입일과 주문일의 순서는 별도 확인이 필요하다.

### Evidence

캡처 첨부 대기: `step02_quality.png`

<!-- ![결측 중복 점검](images/step02_quality.png) -->

## 3. 숫자형·범주형·날짜 점검

### 실행/결과

In [8]:
numeric = pd.concat({"age": customers["age"].describe(),
                     "price": products["price"].describe(),
                     "quantity": order_items["quantity"].describe(),
                     "unit_price": order_items["unit_price"].describe()}, axis=1).round(2)
display(numeric)
for name, df, column in [("customers", customers, "city"),
                         ("products", products, "category"),
                         ("orders", orders, "order_status")]:
    print(name, column)
    display(df[column].value_counts(dropna=False).rename("건수").to_frame())

,age,price,quantity,unit_price
count,150.00,100.00,764.00,764.00
mean,42.09,110040.00,3.05,108561.52
std,15.61,56433.91,1.41,56996.77
min,19.00,5000.00,1.00,5000.00
25%,29.00,65750.00,2.00,62000.00
50%,40.00,112000.00,3.00,111000.00
75%,57.00,161000.00,4.00,161250.00
max,69.00,200000.00,5.00,200000.00


customers city


,건수
city,
성남,21
광주,17
부산,16
대구,15
서울,15
울산,14
인천,14
대전,14
수원,13


products category


,건수
category,
스포츠,19
전자기기,17
생활용품,16
뷰티,16
도서,14
패션,11
식품,7


orders order_status


,건수
order_status,
completed,184
cancelled,64
refunded,52


In [9]:
date_rows = []
for name, df, col in [("customers", customers, "signup_date"), ("orders", orders, "order_date")]:
    raw = df[col].copy()
    converted = pd.to_datetime(raw, errors="coerce")
    date_rows.append({"컬럼": col, "원본 결측": int(raw.isna().sum()),
                      "변환 실패": int((raw.notna() & converted.isna()).sum()),
                      "시작일": str(converted.min().date()), "종료일": str(converted.max().date())})
    df[col] = converted
date_summary = pd.DataFrame(date_rows)

In [10]:
# CAPTURE 3: 범위·주문 상태·날짜 결과 요약
print("숫자형 범위")
display(numeric.loc[["min", "50%", "max"]])
print("주문 상태별 건수")
print(orders["order_status"].value_counts(dropna=False).to_string())
display(date_summary)

숫자형 범위


,age,price,quantity,unit_price
min,19.0,5000.0,1.0,5000.0
50%,40.0,112000.0,3.0,111000.0
max,69.0,200000.0,5.0,200000.0


주문 상태별 건수
order_status
completed    184
cancelled     64
refunded      52


,컬럼,원본 결측,변환 실패,시작일,종료일
0,signup_date,0,0,2023-09-22,2026-09-11
1,order_date,0,0,2025-09-15,2026-09-15


### 결과 관찰

age는 19~69세, 상품 price와 주문 상세 unit_price는 5,000~200,000, quantity는 1~5였다. 고객 연령의 중앙값은 40세이고 주문 상세 수량의 중앙값은 3이었다. 상품은 스포츠가 19개로 가장 많았고 식품은 7개로 가장 적었다.

주문 상태는 completed 184건, cancelled 64건, refunded 52건이었다. 주문일과 가입일의 원본 결측 및 날짜 변환 실패는 각각 0건이었다. 주문 기간은 2025-09-15~2026-09-15, 가입일은 2023-09-22~2026-09-11이었다.

### 나의 해석과 판단

숫자 범위만으로 명백한 오류라고 판단할 값은 찾지 못했다. 다만 최대 가격이 크다는 이유로 이상치를 삭제할 수는 없고 실제 상품 가격 정책을 확인해야 한다. 금액을 분석할 때는 completed 주문과 취소·환불 주문을 구분해야 한다.

### 업무·분석적 의미

주문 상태와 기간 기준을 정하지 않으면 금액과 월별 주문 수가 다르게 계산된다. 기간 양끝의 2025년 9월과 2026년 9월은 일부 날짜만 포함하므로 완전한 월과 단순 비교하지 않아야 한다.

### 한계와 추가 확인 사항

날짜 변환 성공은 형식이 읽힌다는 의미다. 날짜의 업무상 순서까지 보장하지는 않는다. 이번 날짜는 Chapter 02에서 로컬로 생성한 CSV 기준이므로 강의 저장소 예시와 다를 수 있다.

### Evidence

캡처 첨부 대기: `step03_distribution.png`

<!-- ![기본 분포와 날짜 확인](images/step03_distribution.png) -->

## 4. CSV 간 키 관계 검증

### 실행/결과

In [11]:
# CAPTURE 4: 부모 데이터에 존재하지 않는 ID 검사
relationship = pd.DataFrame([
    {"연결": "orders.customer_id → customers.customer_id",
     "미매칭 행 수": int((~orders["customer_id"].isin(customers["customer_id"])).sum())},
    {"연결": "order_items.order_id → orders.order_id",
     "미매칭 행 수": int((~order_items["order_id"].isin(orders["order_id"])).sum())},
    {"연결": "order_items.product_id → products.product_id",
     "미매칭 행 수": int((~order_items["product_id"].isin(products["product_id"])).sum())}
])
display(relationship)
merged = order_items.merge(products, on="product_id", how="left", validate="many_to_one")
print("주문 상세 행 수 / 상품 병합 후 행 수:", len(order_items), "/", len(merged))

,연결,미매칭 행 수
0,orders.customer_id → customers.customer_id,0
1,order_items.order_id → orders.order_id,0
2,order_items.product_id → products.product_id,0


주문 상세 행 수 / 상품 병합 후 행 수: 764 / 764


In [12]:
# 공식 Notebook의 병합 연습: 주문 상태 필터 전 전체 주문상세 금액
merged["line_amount"] = merged["quantity"] * merged["unit_price"]
category_amount = (merged.groupby("category")["line_amount"].sum()
                  .sort_values(ascending=False).rename("전체 주문상세 금액").to_frame())
display(category_amount)
print("이 표는 취소·환불을 포함하며 확정 매출이 아닙니다.")

,전체 주문상세 금액
category,
스포츠,50174000
뷰티,47551000
전자기기,41003000
생활용품,34839000
식품,33597000
도서,24645000
패션,23801000


이 표는 취소·환불을 포함하며 확정 매출이 아닙니다.


### 결과 관찰

부모 customers에 없는 customer_id, 부모 orders에 없는 order_id, 부모 products에 없는 product_id를 참조하는 행은 모두 0건이었다. 상품 정보를 many_to_one 조건으로 병합했을 때도 주문 상세 행 수는 764행으로 유지됐다.

### 나의 해석과 판단

세 관계는 현재 데이터에서 기본적인 연결 조건을 만족한다. 미매칭이 나왔다면 원본 파일 누락, ID 타입, 공백, 수집 시점을 먼저 확인해야 한다. 바로 삭제하면 실제 주문 내역이 사라지고 분석 결과가 바뀔 수 있다.

### 업무·분석적 의미

키 존재 여부와 병합 전후 행 수를 함께 확인하면 연결 누락이나 의도하지 않은 행 증가를 찾을 수 있다. 카테고리별 금액은 병합 확인용으로 계산했으며, 매출 지표에는 주문 상태 처리 기준이 추가로 필요하다.

### 한계와 추가 확인 사항

외래키가 존재해도 그 고객의 가입일과 주문일이 올바른 순서인지는 알 수 없다. 구조적 연결과 업무 규칙은 구분해 검증해야 한다.

### Evidence

캡처 첨부 대기: `step04_relationship.png`

<!-- ![PK FK 관계 검증](images/step04_relationship.png) -->

## 5. LLM 구조 설명 검증

### LLM에 제공한 Safe Context

원본 이름이나 거래 행을 전달하지 않고, 파일별 규모·컬럼·타입과 검증 집계만 제공했다.

In [13]:
safe_prompt = f"""온라인 쇼핑몰의 가상 데이터로 분석 전 점검을 진행하고 있다.
{structure.to_string(index=False)}

정수형: ID, age, price, quantity, unit_price. 나머지는 날짜 변환 전 str.
네 파일의 컬럼 결측, 전체 행 중복, 기본키 결측·중복은 모두 0이다.
세 FK 연결의 미매칭 행 수도 모두 0이다.
연령 19~69, 가격 및 단가 5000~200000, 수량 1~5이다.
주문 상태: completed 184, cancelled 64, refunded 52.
가입일/주문일 변환 실패는 0이고, 가입일 범위는 2023-09-22~2026-09-11,
주문일 범위는 2025-09-15~2026-09-15이다.

분석 전에 추가로 확인할 항목 4개를 '제안 | 현재 확인한 사실 | 추가 검증/주의' 표로 짧게 제시해 줘.
가입일과 주문일의 순서, 범주 표기/공백, 매출의 주문 상태 기준, 병합 행 증가를 포함해 줘.
확인하지 않은 내용이나 원인을 단정하지 마. 개인정보 원본은 사용하지 않는다."""
print(safe_prompt)

온라인 쇼핑몰의 가상 데이터로 분석 전 점검을 진행하고 있다.
         파일   행  열                                                              컬럼
  customers 150  6               customer_id, name, gender, age, city, signup_date
   products 100  4                       product_id, product_name, category, price
     orders 300  5 order_id, customer_id, order_date, payment_method, order_status
order_items 764  5       order_item_id, order_id, product_id, quantity, unit_price

정수형: ID, age, price, quantity, unit_price. 나머지는 날짜 변환 전 str.
네 파일의 컬럼 결측, 전체 행 중복, 기본키 결측·중복은 모두 0이다.
세 FK 연결의 미매칭 행 수도 모두 0이다.
연령 19~69, 가격 및 단가 5000~200000, 수량 1~5이다.
주문 상태: completed 184, cancelled 64, refunded 52.
가입일/주문일 변환 실패는 0이고, 가입일 범위는 2023-09-22~2026-09-11,
주문일 범위는 2025-09-15~2026-09-15이다.

분석 전에 추가로 확인할 항목 4개를 '제안 | 현재 확인한 사실 | 추가 검증/주의' 표로 짧게 제시해 줘.
가입일과 주문일의 순서, 범주 표기/공백, 매출의 주문 상태 기준, 병합 행 증가를 포함해 줘.
확인하지 않은 내용이나 원인을 단정하지 마. 개인정보 원본은 사용하지 않는다.


### LLM이 제안한 추가 점검

| 제안 | 현재 요약으로 확인한 사실 | 추가 검증/주의 |
|---|---|---|
| 가입일·주문일 선후관계 확인 | 날짜 변환 실패는 모두 0건 | 고객별로 연결하여 `order_date < signup_date`인 주문이 있는지 확인해야 함. 아직 미실행 |
| 범주값의 표기·공백 확인 | 문자열 열이 존재하며 주문 상태별 건수가 제공됨 | 성별·도시·카테고리·결제수단·주문 상태의 앞뒤 공백, 대소문자, 철자 차이를 확인해야 함. 아직 미실행 |
| 매출 산정 시 주문 상태 처리 기준 확인 | completed 184건, cancelled 64건, refunded 52건 | 매출에 포함할 상태와 환불 차감 기준을 먼저 정의해야 함. 상태별 건수만으로 매출 처리의 타당성을 판단할 수 없음 |
| 조인 관계와 행 수 증감 확인 | 기본키 누락·중복 및 외래키 불일치는 모두 0건 | 고객→주문→품목의 일대다 관계에서 조인 전후 행 수와 주문별 합계를 대조해야 함. 주문 중복 집계·행 누락 검증은 아직 미실행 |


구조 요약을 제공해 받은 Codex 응답을 기록했다. 아래 코드는 제안을 실제 데이터와 비교한 검증이다.

In [14]:
# CAPTURE 5B: LLM 제안에 대한 실제 검증
order_customer = orders.merge(customers[["customer_id", "signup_date"]],
                              on="customer_id", how="left", validate="many_to_one")
before_signup = int((order_customer["order_date"] < order_customer["signup_date"]).sum())
whitespace_counts = {}
for name, df, col in [("customers", customers, "city"),
                      ("products", products, "category"),
                      ("orders", orders, "order_status")]:
    text_values = df[col].astype("string")
    whitespace_counts[f"{name}.{col}"] = int(text_values.ne(text_values.str.strip()).fillna(False).sum())
checks = pd.DataFrame([
    {"검증": "가입일보다 빠른 주문", "결과": before_signup, "판단": "규칙 확인 전 유지·검토"},
    {"검증": "city/category/order_status 앞뒤 공백", "결과": sum(whitespace_counts.values()), "판단": "공백 문제 미발견"},
    {"검증": "completed 주문 수", "결과": int(orders["order_status"].eq("completed").sum()), "판단": "매출 기준 확정은 별도"},
    {"검증": "상품 병합 전 행 수", "결과": len(order_items), "판단": "many_to_one 확인"},
    {"검증": "상품 병합 후 행 수", "결과": len(merged), "판단": "행 증가 없음"}
])
display(checks)
print("범주별 공백 수:", whitespace_counts)

,검증,결과,판단
0,가입일보다 빠른 주문,48,규칙 확인 전 유지·검토
1,city/category/order_status 앞뒤 공백,0,공백 문제 미발견
2,completed 주문 수,184,매출 기준 확정은 별도
3,상품 병합 전 행 수,764,many_to_one 확인
4,상품 병합 후 행 수,764,행 증가 없음


범주별 공백 수: {'customers.city': 0, 'products.category': 0, 'orders.order_status': 0}


### 실제 데이터에서 확인한 항목

가입일보다 주문일이 빠른 주문이 48건 있었다. city·category·order_status의 앞뒤 공백은 각각 0건이었다. 상품 병합 전후 행 수는 764행으로 같았고 completed 주문은 184건이었다.

### 채택·수정·보류한 내용

| 제안 | 판단 | 근거 |
|---|---|---|
| 가입일과 주문일의 순서 확인 | 채택 | 형식 변환은 성공했지만 날짜 순서가 다른 주문 48건을 발견했다. |
| 범주 표기·공백 확인 | 채택, 검증 범위 제한 | 앞뒤 공백은 없었다. 값별 빈도도 확인했지만 공식 허용값 목록이 없어 의미상 오타까지 정상이라고 단정하지 않았다. |
| 매출 상태 기준 정의 | 수정·일부 보류 | completed만 따로 확인했다. 현재 카테고리 금액 표는 취소·환불 포함값이므로 매출이라고 부르지 않았다. 환불·비용 처리의 업무 기준은 별도로 확인해야 한다. |
| 병합으로 행이 늘어나는지 확인 | 채택 | many_to_one 검증과 행 수 비교에서 상품 병합 후 증가가 없었다. |

### 나의 해석과 판단

가입일과 주문일을 비교하라는 제안이 가장 유용했다. 결측과 키 오류가 없어도 날짜 순서에는 문제가 있을 수 있었다. 다만 48건을 곧바로 잘못된 데이터로 삭제하지 않았다. 가상 데이터 생성 방식이나 비회원 주문, 데이터 이관 같은 업무 규칙을 확인한 뒤 처리해야 한다고 판단했다. 구조가 연결된다는 이유만으로 바로 매출 분석이 가능하다고 단정하는 설명은 주의해야 한다.

### 한계와 추가 확인 사항

비회원 주문이나 데이터 이관 여부는 현재 CSV만으로 확인할 수 없다. 48건이 발생한 원인을 확정하지 않았고, 원본 행도 수정하지 않았다. LLM 답변은 점검 제안이며 사실 판단은 실행 결과를 기준으로 했다.

### Evidence

캡처 첨부 대기: `step05_llm.png`

<!-- ![LLM 구조 검토 대화](images/step05_llm.png) -->

### Evidence

캡처 첨부 대기: `step05_validation.png`

<!-- ![LLM 제안의 실제 검증](images/step05_validation.png) -->

## 6. Chapter 03 최종 판단

### 데이터의 첫인상 3가지

1. 고객·상품·주문·주문 상세의 역할이 분리돼 있고, 네 CSV의 결측·기본키 중복·외래키 미매칭은 모두 0건이었다.
2. 주문 상세의 order_id 반복은 주문 안의 여러 상품을 나타내므로 삭제 대상이 아니었다. 날짜는 문자열에서 날짜 타입으로 변환해야 했다.
3. 형식과 키가 정상이어도 업무상 순서까지 정상은 아니었다. 가입일보다 빠른 주문 48건을 확인해 후속 점검 대상으로 남겼다.

### 다음 Chapter 전에 반드시 확인/처리해야 할 항목

1. 날짜 순서가 맞지 않는 48건의 생성 방식과 업무 정의를 확인하고, 포함 여부에 따른 분석 차이를 검토한다.
2. 금액을 매출로 사용할 때 completed·cancelled·refunded 처리 기준과 가격의 단위를 확인한다.
3. 월별 비교에서 불완전한 시작·종료월을 구분하고, 병합 키의 관계와 행 수를 다시 확인한다.

### 현재 데이터만으로 단정할 수 없는 것

이 가상 데이터가 실제 고객 행동을 대표하는지, 특정 카테고리가 많이 판매되는 이유, 취소·환불 원인, 가입일 이전 주문의 원인은 단정할 수 없다. 컬럼과 값의 형식만으로 업무 정의가 충분하다고 판단하지 않았다.

## 최종 제출 체크

- [x] 코드 셀을 처음부터 끝까지 실행해 결과를 저장했다.
- [x] 저장된 실행 결과에 오류 셀이 없다.
- [ ] VS Code에서 직접 재실행하고 결과를 확인했다.
- [ ] 핵심 Evidence를 첨부했다.
- [x] 관찰과 해석을 구분했다.
- [x] 실제 개인정보와 Secret을 포함하지 않았다.
- [ ] Notebook과 Evidence의 GitHub 표시를 최종 확인했다.
- [ ] 최종 Notebook 파일 URL을 LMS에 제출했다.